In [12]:
!pip install imagehash

In [13]:
import torch
import torchvision
import sklearn
import pandas as pd
import matplotlib as plt
import seaborn as sb
import os
import shutil
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split
from collections import defaultdict

In [14]:
import os
from PIL import Image
import imagehash
from collections import Counter

def audit_dataset(base_dir=".", hamming_threshold=5):
    splits = ["train", "test", "unclean"]
    all_images = []
    class_counts = {split: Counter() for split in splits}
    unreadable_files = []
    unusual_sizes = []

    for split in splits:
        split_dir = os.path.join(base_dir, split)
        if not os.path.exists(split_dir):
            print(f"Warning: Directory {split_dir} not found. Please ensure the dataset is extracted correctly.")
            continue
            
        print(f"Scanning split: {split}...")
        for root, _, files in os.walk(split_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    img_path = os.path.join(root, file)
                    
                    # استخراج نام کلاس از روی پوشه والد
                    rel_path = os.path.relpath(img_path, split_dir)
                    path_parts = rel_path.split(os.sep)
                    if len(path_parts) > 1:
                        class_name = path_parts[0]
                        class_counts[split][class_name] += 1

                    try:
                        with Image.open(img_path) as img:
                            width, height = img.size
                            
                            # بررسی تصاویر غیرعادی کوچک (کمتر از 32 در 32 پیکسل)
                            if width < 32 or height < 32:
                                unusual_sizes.append((img_path, (width, height)))

                            # محاسبه هش ادراکی به صورت شیء (برای محاسبه فاصله همینگ)
                            img_hash = imagehash.phash(img)
                            full_rel_path = os.path.relpath(img_path, base_dir)
                            all_images.append((full_rel_path, img_hash))
                            
                    except Exception as e:
                        unreadable_files.append((img_path, str(e)))

    # پیدا کردن تصاویر مشابه یا تکراری با استفاده از فاصله همینگ
    duplicates = []
    visited = set()
    
    for i in range(len(all_images)):
        if i in visited:
            continue
        group = [all_images[i][0]]
        for j in range(i + 1, len(all_images)):
            if j in visited:
                continue
            # محاسبه فاصله همینگ بین دو هش (- روی شیءهای imagehash فاصله همینگ را می‌دهد)
            hamming_dist = all_images[i][1] - all_images[j][1]
            if hamming_dist <= hamming_threshold:
                group.append(all_images[j][0])
                visited.add(j)
        if len(group) > 1:
            duplicates.append(group)

    # چاپ گزارش نهایی
    print("\n" + "="*60)
    print("--- DATASET AUDIT REPORT (WITH HAMMING DISTANCE) ---")
    print("="*60)
    
    print(f"\n1. Unreadable or corrupted files: {len(unreadable_files)}")
    for path, err in unreadable_files:
        print(f"   - {path}: {err}")

    print(f"\n2. Unusually small images (< 32x32): {len(unusual_sizes)}")
    for path, size in unusual_sizes[:5]:
        print(f"   - {path} (Size: {size})")

    print("\n3. Class distribution per split:")
    for split, counts in class_counts.items():
        print(f"   [{split}]:")
        for cls, cnt in counts.items():
            print(f"     - {cls}: {cnt}")

    print(f"\n4. Similar/Duplicate groups found (Threshold <= {hamming_threshold}): {len(duplicates)}")
    for idx, group in enumerate(duplicates[:10]):
        print(f"   Group {idx + 1}:")
        for p in group:
            print(f"     * {p}")
    if len(duplicates) > 10:
        print(f"   ... and {len(duplicates) - 10} more groups.")

    print("\nDataset audit with Hamming distance completed successfully!")

if __name__ == "__main__":
    audit_dataset()

Scanning split: train...
Scanning split: test...
Scanning split: unclean...

--- DATASET AUDIT REPORT (WITH HAMMING DISTANCE) ---

1. Unreadable or corrupted files: 0

2. Unusually small images (< 32x32): 0

3. Class distribution per split:
   [train]:
     - ambulance: 50
     - autobus: 50
     - kamyun: 50
     - kamyunet: 50
     - minibus: 50
     - savari: 50
     - taxi: 50
     - vanet: 50
   [test]:
     - ambulance: 50
     - autobus: 50
     - kamyun: 50
     - kamyunet: 50
     - minibus: 50
     - savari: 50
     - taxi: 50
     - vanet: 50
   [unclean]:
     - ambulance: 50
     - autobus: 50
     - kamyun: 50
     - kamyunet: 50
     - minibus: 50
     - neysan: 50
     - savari: 50
     - taxi: 50
     - vanet: 50

4. Similar/Duplicate groups found (Threshold <= 5): 29
   Group 1:
     * train\ambulance\199984667.jpg
     * unclean\ambulance\199984667.jpg
   Group 2:
     * train\ambulance\206808986.jpg
     * unclean\ambulance\206808986.jpg
   Group 3:
     * train\amb

# cleaning dataset

In [15]:


def clean_and_split_dataset(base_dir=".", output_dir="dataset_cleaned", hamming_threshold=5, test_size=0.2, random_seed=42):
    splits = ["train", "test", "unclean"]
    all_records = []
    
    print("1. Scanning and collecting image records...")
    for split in splits:
        split_dir = os.path.join(base_dir, split)
        if not os.path.exists(split_dir):
            continue
            
        for root, _, files in os.walk(split_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    img_path = os.path.join(root, file)
                    rel_path = os.path.relpath(img_path, base_dir)
                    path_parts = rel_path.split(os.sep)
                    
                    if len(path_parts) > 1:
                        class_name = path_parts[1] # نام کلاس
                        
                        # کلاس نیسان را برای آموزش/اعتبارسنجی وارد نکنیم (طبق دستورالعمل)
                        if class_name == "neysan":
                            continue
                            
                        try:
                            with Image.open(img_path) as img:
                                img_hash = imagehash.phash(img)
                                all_records.append({
                                    "path": img_path,
                                    "class": class_name,
                                    "hash": img_hash,
                                    "split": split
                                })
                        except Exception as e:
                            print(f"Skipping unreadable file {img_path}: {e}")

    print(f"Total valid images collected (excluding neysan): {len(all_records)}")

    print("2. Removing duplicates using Hamming distance...")
    drop_indices = set()
    for i in range(len(all_records)):
        if i in drop_indices:
            continue
        for j in range(i + 1, len(all_records)):
            if j in drop_indices:
                continue
            # اگر تصاویر بسیار شبیه یا تکراری بودند، یکی را حذف می‌کنیم
            if all_records[i]["hash"] - all_records[j]["hash"] <= hamming_threshold:
                # اولویت نگهداری با train است، بعد test، بعد unclean
                drop_indices.add(j)

    clean_records = [rec for idx, rec in enumerate(all_records) if idx not in drop_indices]
    print(f"Images remaining after duplicate removal: {len(clean_records)}")

    # تفکیک بر اساس کلاس و برچسب برای تقسیم Stratified
    paths = [rec["path"] for rec in clean_records]
    labels = [rec["class"] for rec in clean_records]

    print("3. Creating 80/20 stratified train/validation split...")
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        paths, labels, test_size=test_size, random_state=random_seed, stratify=labels
    )

    # تابع کمکی برای کپی کردن فایل‌ها به ساختار جدید
    def save_to_split(file_paths, target_split_name):
        for path in file_paths:
            # استخراج نام کلاس و نام فایل
            parts = path.split(os.sep)
            class_name = parts[-2]
            file_name = parts[-1]
            
            dest_dir = os.path.join(output_dir, target_split_name, class_name)
            os.makedirs(dest_dir, exist_ok=True)
            shutil.copy(path, os.path.join(dest_dir, file_name))

    # ذخیره در پوشه جدید
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        
    save_to_split(train_paths, "train")
    save_to_split(val_paths, "val")

    print(f"\nDataset successfully cleaned and split into '{output_dir}/train' and '{output_dir}/val'!")
    print(f"Training samples: {len(train_paths)}")
    print(f"Validation samples: {len(val_paths)}")

if __name__ == "__main__":
    clean_and_split_dataset()

1. Scanning and collecting image records...
Total valid images collected (excluding neysan): 1200
2. Removing duplicates using Hamming distance...
Images remaining after duplicate removal: 1166
3. Creating 80/20 stratified train/validation split...

Dataset successfully cleaned and split into 'dataset_cleaned/train' and 'dataset_cleaned/val'!
Training samples: 932
Validation samples: 234
